## 🥈 Silver Layer – Data Cleaning & Transformation

### 🎯 Objective

The Silver layer refines raw Bronze data into a clean, consistent, and analytics-ready dataset.

---

### 🔧 Transformations Applied

1. **Deduplication**

   * Removed duplicate records based on `order_id` and `timestamp`

2. **Null Handling**

   * Dropped records with critical null fields (`order_id`, `price`, `quantity`)

3. **Standardization**

   * Normalized `region` values to lowercase and trimmed whitespace

4. **Derived Column**

   * Created `revenue = quantity * price`

5. **Data Type Corrections**

   * Converted `timestamp` to proper timestamp format

---

### 📌 Outcome

This layer ensures:

* Data consistency
* Improved data quality
* Readiness for business analytics in Gold layer


In [0]:
from pyspark.sql.functions import lower, trim, col, to_timestamp


In [0]:
df_bronze = spark.read.table("workspace.smartgear.bronze_orders")

In [0]:
df_bronze.display()

order_id,order_number,timestamp,store_id,product,quantity,price,region,ingestion_time,source
a8980701-75b8-46f9-9e8a-c5e854b8dbce,2001,2026-04-26T11:11:42.767786,101,Monitor,5,213.79,North,2026-04-26T11:15:37.111Z,kafka
4b1ccd97-3e76-4aa9-931d-2c7fc5e10fe0,2001,2026-04-26T11:11:42.768081,107,Camera,1,498.16,West,2026-04-26T11:15:37.111Z,kafka
3ee28a0e-36b7-47a3-9220-e4c8d1fc28b4,2001,2026-04-26T11:11:42.768109,102,Drone,2,674.87,East,2026-04-26T11:15:37.111Z,kafka
2e6053f2-ee81-4130-8610-7858d1ee7571,2001,2026-04-26T11:11:42.768157,108,Camera,2,548.54,West,2026-04-26T11:15:37.111Z,kafka
34021dc5-178c-41f0-a298-7124c99d5460,2001,2026-04-26T11:11:48.458544,111,Smartwatch,4,264.23,South,2026-04-26T11:15:37.111Z,kafka
36d22c75-4493-4ce7-a2db-0ba5c22ac5b1,2001,2026-04-26T11:11:48.458629,108,Tablet,5,382.47,East,2026-04-26T11:15:37.111Z,kafka
5415561c-3379-44fa-abd7-1c097072e7ef,2001,2026-04-26T11:11:48.458819,104,Headphones,2,87.2,South,2026-04-26T11:15:37.111Z,kafka
83f61eb9-f52f-4f94-8e7c-a1d517ef0196,2001,2026-04-26T11:11:42.767969,109,Gaming Console,5,443.05,North,2026-04-26T11:15:37.111Z,kafka
70c9e4c1-0371-4a3d-9828-2f6f29e0cf30,2001,2026-04-26T11:11:42.768044,119,Laptop,5,806.84,East,2026-04-26T11:15:37.111Z,kafka
77738df7-30a8-4379-916a-dfa65cff507d,2001,2026-04-26T11:11:42.768134,115,Camera,1,498.14,West,2026-04-26T11:15:37.111Z,kafka


In [0]:
#Removing Duplicates
df_dedup = df_bronze.dropDuplicates(["order_id", "timestamp"])

#Removing NULL rows
df_clean = df_dedup.dropna(subset=["order_id", "price", "quantity"])

#Standardize Region
#Calculating Revenue Column
#Fixing Timestamp Type
df_clean = df_clean \
    .withColumn("region", lower(trim(col("region")))) \
    .withColumn("revenue", col("quantity") * col("price")) \
    .withColumn("timestamp", to_timestamp("timestamp"))

In a production environment, this layer would be implemented using incremental processing (MERGE or append + deduplication) instead of full overwrite to ensure scalability and efficiency.

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.smartgear.silver_orders")

In [0]:
%sql
SELECT * FROM workspace.smartgear.silver_orders;

order_id,order_number,timestamp,store_id,product,quantity,price,region,ingestion_time,source,revenue
5415561c-3379-44fa-abd7-1c097072e7ef,2001,2026-04-26T11:11:48.458Z,104,Headphones,2,87.2,south,2026-04-26T11:15:37.111Z,kafka,174.4
36d22c75-4493-4ce7-a2db-0ba5c22ac5b1,2001,2026-04-26T11:11:48.458Z,108,Tablet,5,382.47,east,2026-04-26T11:15:37.111Z,kafka,1912.3500000000001
4b1ccd97-3e76-4aa9-931d-2c7fc5e10fe0,2001,2026-04-26T11:11:42.768Z,107,Camera,1,498.16,west,2026-04-26T11:15:37.111Z,kafka,498.16
2e6053f2-ee81-4130-8610-7858d1ee7571,2001,2026-04-26T11:11:42.768Z,108,Camera,2,548.54,west,2026-04-26T11:15:37.111Z,kafka,1097.08
a8980701-75b8-46f9-9e8a-c5e854b8dbce,2001,2026-04-26T11:11:42.767Z,101,Monitor,5,213.79,north,2026-04-26T11:15:37.111Z,kafka,1068.95
3ee28a0e-36b7-47a3-9220-e4c8d1fc28b4,2001,2026-04-26T11:11:42.768Z,102,Drone,2,674.87,east,2026-04-26T11:15:37.111Z,kafka,1349.74
34021dc5-178c-41f0-a298-7124c99d5460,2001,2026-04-26T11:11:48.458Z,111,Smartwatch,4,264.23,south,2026-04-26T11:15:37.111Z,kafka,1056.92
70c9e4c1-0371-4a3d-9828-2f6f29e0cf30,2001,2026-04-26T11:11:42.768Z,119,Laptop,5,806.84,east,2026-04-26T11:15:37.111Z,kafka,4034.2000000000003
77738df7-30a8-4379-916a-dfa65cff507d,2001,2026-04-26T11:11:42.768Z,115,Camera,1,498.14,west,2026-04-26T11:15:37.111Z,kafka,498.14
dbf45810-4d65-4eb0-bec3-600f051b9117,2001,2026-04-26T11:11:48.458Z,109,Camera,4,537.22,south,2026-04-26T11:15:37.111Z,kafka,2148.88
